# Redis Complete Course: From Basics to Production

A comprehensive guide covering Redis fundamentals, advanced data structures, patterns, performance optimization, and FastAPI integration.

---

## Table of Contents

1. **Module 1: Redis Fundamentals**
   - What is Redis?
   - Installation & Setup
   - Basic Connection

2. **Module 2: Data Structures & Operations**
   - Strings
   - Lists
   - Sets
   - Hashes
   - Sorted Sets

3. **Module 3: Advanced Concepts**
   - Key Expiration & TTL
   - Transactions
   - Pub/Sub Messaging
   - Pipelining

4. **Module 4: Design Patterns**
   - Caching Strategies
   - Rate Limiting
   - Session Management
   - Distributed Locks

5. **Module 5: Performance & Optimization**
   - Key Design
   - Memory Management
   - Persistence
   - Monitoring

6. **Module 6: FastAPI Integration**
   - Setting up Redis with FastAPI
   - Caching API Responses
   - Rate Limiting Middleware
   - Real-time Features

---
# Module 1: Redis Fundamentals

## 1.1 What is Redis?

**Redis** (Remote Dictionary Server) is an in-memory data structure store that:
- Operates as a key-value database
- Runs entirely in RAM for blazing-fast access
- Supports multiple data structures (strings, lists, sets, etc.)
- Provides persistence options (RDB, AOF)
- Enables pub/sub messaging
- Ideal for caching, sessions, real-time analytics, rate limiting

### Common Use Cases:
- **Caching Layer**: Reduce database load
- **Session Store**: Fast session management
- **Rate Limiting**: Track request counts
- **Leaderboards**: Sorted ranked data
- **Real-time Analytics**: Quick data aggregation
- **Pub/Sub Messaging**: Event-driven systems

## 1.2 Installation & Setup

### Prerequisites
- Redis server running (Windows, macOS, or Linux)
- Python 3.7+
- redis-py library

### Installation Commands

In [1]:
# Install redis-py client library
# Run this in your terminal:
# pip install redis

# For Windows: Download Redis from https://github.com/microsoftarchive/redis/releases
# For macOS: brew install redis
# For Linux: sudo apt-get install redis-server

# Start Redis server:
# Windows: redis-server.exe
# macOS/Linux: redis-server

print("Installation steps completed. Proceed to the next section.")

Installation steps completed. Proceed to the next section.


## 1.3 Basic Connection

In [2]:
# Import Redis client library
import redis
from redis import Redis
import time
import json
from datetime import datetime, timedelta

# Create a Redis connection
# Default: localhost, port 6379
r = redis.Redis(
    host='localhost',      # Redis server address
    port=6379,             # Default Redis port
    db=0,                  # Database number (0-15)
    decode_responses=True  # Automatically decode responses to strings
)

# Test connection
try:
    pong = r.ping()
    print(f"✓ Connected to Redis: {pong}")
except Exception as e:
    print(f"✗ Connection failed: {e}")

✓ Connected to Redis: True


In [3]:
# Get server info
info = r.info()
print(f"Redis Version: {info.get('redis_version')}")
print(f"Used Memory: {info.get('used_memory_human')}")
print(f"Connected Clients: {info.get('connected_clients')}")
print(f"Total Commands Processed: {info.get('total_commands_processed')}")

Redis Version: 8.8.0
Used Memory: 2.77M
Connected Clients: 1
Total Commands Processed: 1002


---
# Module 2: Data Structures & Operations

## 2.1 Strings - The Basic Data Type

Strings are the simplest Redis data structure. They can store text, numbers, or binary data.
Maximum size: 512 MB

In [4]:
# ===== SET AND GET =====
# SET: Store a value
r.set('username', 'alice')
r.set('age', 25)

# GET: Retrieve a value
username = r.get('username')
age = r.get('age')

print(f"Username: {username}")
print(f"Age: {age}")
print(f"Age type: {type(age)}")

Username: alice
Age: 25
Age type: <class 'str'>


In [5]:
# ===== MSET AND MGET (Multiple SET/GET) =====
# MSET: Set multiple key-value pairs at once
r.mset({
    'email': 'alice@example.com',
    'country': 'USA',
    'city': 'New York'
})

# MGET: Get multiple values
values = r.mget('email', 'country', 'city')
print(f"Multiple values: {values}")

Multiple values: ['alice@example.com', 'USA', 'New York']


In [6]:
r.mget('email', 'country', 'city', 'nonexistent_key')  # Returns None for nonexistent keys

['alice@example.com', 'USA', 'New York', None]

In [7]:
r.mget('email', 'country','age')

['alice@example.com', 'USA', '25']

In [8]:
# ===== APPEND =====
# APPEND: Add text to existing value
r.set('greeting', 'Hello')
r.append('greeting', ' World!')
print(f"After append: {r.get('greeting')}")

After append: Hello World!


In [9]:
# ===== INCR AND DECR (Atomic increment/decrement) =====
# INCR: Increment a number by 1
r.set('counter', 10)
r.incr('counter')  # Now: 11
r.incr('counter')  # Now: 12
print(f"Counter after INCR: {r.get('counter')}")

# INCRBY: Increment by specific value
r.incrby('counter', 5)  # Now: 17
print(f"Counter after INCRBY 5: {r.get('counter')}")

# DECR: Decrement by 1
r.decr('counter')  # Now: 16
print(f"Counter after DECR: {r.get('counter')}")

Counter after INCR: 12
Counter after INCRBY 5: 17
Counter after DECR: 16


In [10]:
# ===== GETRANGE AND SETRANGE =====
# GETRANGE: Get substring
r.set('message', 'Hello World')
substring = r.getrange('message', 0, 4)  # Chars 0-4
print(f"GETRANGE (0, 4): {substring}")

# SETRANGE: Replace substring
r.setrange('message', 6, 'Redis')  # Replace from position 6
print(f"After SETRANGE: {r.get('message')}")

GETRANGE (0, 4): Hello
After SETRANGE: Hello Redis


## 2.2 Lists - Ordered Collections

Lists are ordered collections of strings. Elements maintain insertion order.
Can be used as queues or stacks.

In [11]:
# ===== PUSH AND POP =====
# RPUSH: Push to the right (end) of list
r.delete('tasks')  # Clear previous data
r.rpush('tasks', 'Task 1', 'Task 2', 'Task 3')
print(f"After RPUSH: {r.lrange('tasks', 0, -1)}")

# LPUSH: Push to the left (start) of list
r.lpush('tasks', 'Priority Task')
print(f"After LPUSH: {r.lrange('tasks', 0, -1)}")

# RPOP: Pop from right (end)
popped_right = r.rpop('tasks')
print(f"Popped from right: {popped_right}")

# LPOP: Pop from left (start)
popped_left = r.lpop('tasks')
print(f"Popped from left: {popped_left}")

After RPUSH: ['Task 1', 'Task 2', 'Task 3']
After LPUSH: ['Priority Task', 'Task 1', 'Task 2', 'Task 3']
Popped from right: Task 3
Popped from left: Priority Task


In [12]:
# viewing the remaining list
print(f"Remaining tasks: {r.lrange('tasks', 0, -1)}")

Remaining tasks: ['Task 1', 'Task 2']


In [13]:
# ===== LRANGE AND LINDEX =====
r.delete('queue')
r.rpush('queue', 'job1', 'job2', 'job3', 'job4', 'job5')
r.lpush('queue', 'urgent_job')  # Add an urgent job at the start
# LRANGE: Get range of elements
all_jobs = r.lrange('queue', 0, -1)  # Get all
print(f"All jobs: {all_jobs}")

first_three = r.lrange('queue', 0, 2)  # First 3
print(f"First 3 jobs: {first_three}")

# LINDEX: Get single element by index
second_job = r.lindex('queue', 1)
print(f"Second job (index 1): {second_job}")
last_job = r.lindex('queue', -1)  # -1 means last
print(f"Last job: {last_job}")

All jobs: ['urgent_job', 'job1', 'job2', 'job3', 'job4', 'job5']
First 3 jobs: ['urgent_job', 'job1', 'job2']
Second job (index 1): job1
Last job: job5


In [14]:
# ===== LIST LENGTH AND OPERATIONS =====
# LLEN: Get list length
length = r.llen('queue')
print(f"Queue length: {length}")

# LSET: Set value at index
r.lset('queue', 0, 'job_updated')
print(f"After LSET: {r.lrange('queue', 0, -1)}")

# LTRIM: Keep only specified range
r.ltrim('queue', 0, 2)  # Keep only first 3
print(f"After LTRIM (0, 2): {r.lrange('queue', 0, -1)}")

Queue length: 6
After LSET: ['job_updated', 'job1', 'job2', 'job3', 'job4', 'job5']
After LTRIM (0, 2): ['job_updated', 'job1', 'job2']


## 2.3 Sets - Unordered Unique Collections

Sets are collections of unique, unordered strings.
Perfect for tracking unique items and set operations.

In [15]:
# ===== SADD AND SMEMBERS =====
# SADD: Add members to set
r.delete('languages')
r.sadd('languages', 'Python', 'JavaScript', 'Java', 'Go')

# SMEMBERS: Get all members (returns unordered)
langs = r.smembers('languages')
print(f"Languages: {langs}")

# Try adding duplicate
result = r.sadd('languages', 'Python')  # Returns 0 (not added)
print(f"Adding duplicate 'Python' returned: {result}")
print(f"Still {r.scard('languages')} unique languages")

result = r.sadd('languages', 'C')  # Returns 1 ( added)
print(f"Adding 'C' returned: {result}")
print(f"Still {r.scard('languages')} unique languages")

Languages: {'Python', 'JavaScript', 'Java', 'Go'}
Adding duplicate 'Python' returned: 0
Still 4 unique languages
Adding 'C' returned: 1
Still 5 unique languages


In [16]:
# ===== SET OPERATIONS =====
# Add second set for operations
r.delete('frontend_langs')
r.sadd('frontend_langs', 'JavaScript', 'TypeScript', 'Go')

# SINTER: Intersection (common elements)
common = r.sinter('languages', 'frontend_langs')
print(f"Common languages: {common}")

# SUNION: Union (all unique elements)
all_langs = r.sunion('languages', 'frontend_langs')
print(f"All languages: {all_langs}")

# SDIFF: Difference (in first set but not in second)
backend_only = r.sdiff('languages', 'frontend_langs')
print(f"Backend-only languages: {backend_only}")

Common languages: {'JavaScript', 'Go'}
All languages: {'JavaScript', 'Java', 'TypeScript', 'C', 'Go', 'Python'}
Backend-only languages: {'Java', 'Python', 'C'}


In [17]:
# ===== SET MEMBERSHIP AND REMOVAL =====
# SISMEMBER: Check if member exists
is_python = r.sismember('languages', 'Python')
is_c = r.sismember('languages', 'C')
print(f"Is 'Python' in languages? {is_python}")
print(f"Is 'C' in languages? {is_c}")

# SREM: Remove member
print(f"Before removing 'Go': {r.smembers('languages')}")
r.srem('languages', 'Go')
print(f"After removing 'Go': {r.smembers('languages')}")

# SCARD: Count members
count = r.scard('languages')
print(f"Set size: {count}")

Is 'Python' in languages? 1
Is 'C' in languages? 1
Before removing 'Go': {'JavaScript', 'Java', 'C', 'Go', 'Python'}
After removing 'Go': {'Python', 'JavaScript', 'C', 'Java'}
Set size: 4


## 2.4 Hashes - Structured Objects

Hashes store multiple field-value pairs under a single key.
Perfect for representing objects or complex data.

In [18]:
# ===== HSET AND HGET =====
# HSET: Set field in hash
r.delete('user:1')
r.hset('user:1', mapping={
    'name': 'Alice',
    'email': 'alice@example.com',
    'age': 28,
    'city': 'New York'
})

# HGET: Get single field
name = r.hget('user:1', 'name')
email = r.hget('user:1', 'email')
print(f"Name: {name}")
print(f"Email: {email}")

Name: Alice
Email: alice@example.com


In [19]:
# ===== HGETALL AND HKEYS/HVALS =====
# HGETALL: Get entire hash
user = r.hgetall('user:1')
print(f"Full user data: {user}")

# HKEYS: Get all field names
fields = r.hkeys('user:1')
print(f"Fields: {fields}")

# HVALS: Get all values
values = r.hvals('user:1')
print(f"Values: {values}")

Full user data: {'name': 'Alice', 'email': 'alice@example.com', 'age': '28', 'city': 'New York'}
Fields: ['name', 'email', 'age', 'city']
Values: ['Alice', 'alice@example.com', '28', 'New York']


In [20]:
# ===== HMGET AND HEXISTS =====
# HMGET: Get multiple fields
user_info = r.hmget('user:1', 'name', 'age', 'city')
print(f"Name, Age, City: {user_info}")

# HEXISTS: Check if field exists
has_phone = r.hexists('user:1', 'phone')
has_name = r.hexists('user:1', 'name')
print(f"Has 'phone' field? {has_phone}")
print(f"Has 'name' field? {has_name}")

# HLEN: Count fields in hash
count = r.hlen('user:1')
print(f"Number of fields: {count}")

Name, Age, City: ['Alice', '28', 'New York']
Has 'phone' field? False
Has 'name' field? True
Number of fields: 4


In [21]:
# ===== HINCRBY AND HDEL =====
# HINCRBY: Increment numeric field
r.hincrby('user:1', 'age', 1)  # Birthday!
print(f"Age after birthday: {r.hget('user:1', 'age')}")

# HDEL: Delete field
r.hdel('user:1', 'city')
print(f"After deleting 'city': {r.hgetall('user:1')}")

Age after birthday: 29
After deleting 'city': {'name': 'Alice', 'email': 'alice@example.com', 'age': '29'}


## 2.5 Sorted Sets - Ranked Collections

Sorted sets store unique members with associated scores.
Members are automatically ranked by score.
Perfect for leaderboards, rankings, and priority queues.

In [22]:
# ===== ZADD AND ZRANGE =====
# ZADD: Add members with scores
r.delete('leaderboard')
r.zadd('leaderboard', {
    'Alice': 100,
    'Bob': 85,
    'Charlie': 110,
    'Diana': 95
})

# ZRANGE: Get members by rank (low to high)
ranks = r.zrange('leaderboard', 0, -1)
print(f"Leaderboard (ascending): {ranks}")

# ZREVRANGE: Get members in reverse (high to low)
top_performers = r.zrevrange('leaderboard', 0, -1)
print(f"Leaderboard (descending): {top_performers}")

Leaderboard (ascending): ['Bob', 'Diana', 'Alice', 'Charlie']
Leaderboard (descending): ['Charlie', 'Alice', 'Diana', 'Bob']


In [23]:
# ===== ZRANGE WITH SCORES =====
# Get members with their scores
rankings = r.zrevrange('leaderboard', 0, -1, withscores=True)
print(f"Rankings :: {rankings}")
print(f"Rankings with scores:\n")
for rank, (player, score) in enumerate(rankings, 1):
    print(f"  {rank}. {player}: {int(score)} points")

Rankings :: [('Charlie', 110.0), ('Alice', 100.0), ('Diana', 95.0), ('Bob', 85.0)]
Rankings with scores:

  1. Charlie: 110 points
  2. Alice: 100 points
  3. Diana: 95 points
  4. Bob: 85 points


In [24]:
# ===== ZSCORE AND ZRANK =====
# ZSCORE: Get score of member
alice_score = r.zscore('leaderboard', 'Alice')
print(f"Alice's score: {alice_score}")

# ZRANK: Get rank of member (0-indexed, low to high)
alice_rank = r.zrank('leaderboard', 'Alice')
print(f"Alice's rank (ascending): {alice_rank + 1}")  # +1 for 1-indexed display

# ZREVRANK: Get reverse rank (high to low)
alice_rev_rank = r.zrevrank('leaderboard', 'Alice')
print(f"Alice's rank (descending): {alice_rev_rank + 1}")

Alice's score: 100.0
Alice's rank (ascending): 3
Alice's rank (descending): 2


In [25]:
# ===== ZINCRBY AND ZCARD =====
# ZINCRBY: Increment member's score
r.zincrby('leaderboard', 10, 'Bob')  # Bob gains 10 points
bob_new_score = r.zscore('leaderboard', 'Bob')
print(f"Bob's new score: {bob_new_score}")

# ZCARD: Count members
total_members = r.zcard('leaderboard')
print(f"Total members: {total_members}")

Bob's new score: 95.0
Total members: 4


In [26]:
# ===== ZCOUNT AND ZRANGEBYSCORE =====
# ZCOUNT: Count members in score range
high_scorers = r.zcount('leaderboard', 95, 120)  # Score 95-120
print(f"Players with score 95-120: {high_scorers}")

# ZRANGEBYSCORE: Get members in score range
members_in_range = r.zrangebyscore('leaderboard', 90, 110, withscores=True)
print(f"Members with score 90-110: {members_in_range}")

Players with score 95-120: 4
Members with score 90-110: [('Bob', 95.0), ('Diana', 95.0), ('Alice', 100.0), ('Charlie', 110.0)]


---
# Module 3: Advanced Concepts

## 3.1 Key Expiration & TTL

Set automatic expiration times on keys. Useful for:
- Temporary cache data
- Session tokens
- Rate limiting counters

In [27]:
r.set('otp', '123456',ex=5)
print(f"OTP set with expiration. Current value: {r.get('otp')}")
print(f"Temp token exists :: {r.exists('otp')}")
time.sleep(11)  # Wait for expiration
print(f"OTP after expiration: {r.get('otp')}")
print(f"Temp token exists :: {r.exists('otp')}")

OTP set with expiration. Current value: 123456
Temp token exists :: 1
OTP after expiration: None
Temp token exists :: 0


In [28]:
# ===== SET WITH EXPIRATION =====
# Set key with 10-second expiration
r.set('temp_session', 'session_xyz', ex=10)  # ex=expiry in seconds

# Alternative: Set then expire
r.set('temp_token', 'token_abc')
r.expire('temp_token', 5)  # Expire in 5 seconds

print(f"Temp session exists: {r.exists('temp_session')}")
print(f"Temp token exists: {r.exists('temp_token')}")

Temp session exists: 1
Temp token exists: 1


In [29]:
# ===== TTL AND PTTL =====
# TTL: Get remaining time to live in seconds
ttl_session = r.ttl('temp_session')
print(f"Temp session TTL: {ttl_session} seconds")

# PTTL: Get TTL in milliseconds
pttl_session = r.pttl('temp_session')
print(f"Temp session PTTL: {pttl_session} milliseconds")

# PERSIST: Remove expiration
r.persist('temp_session')
ttl_after_persist = r.ttl('temp_session')
print(f"TTL after PERSIST: {ttl_after_persist} (-1 means no expiration)")

Temp session TTL: 10 seconds
Temp session PTTL: 9982 milliseconds
TTL after PERSIST: -1 (-1 means no expiration)


In [30]:
r.set("tesing_otp","1234567890",ex=10)
while r.exists('tesing_otp'):
    print(f"Testing OTP expires in : {r.ttl('tesing_otp')}")
    time.sleep(1)  # Check every 1 second

Testing OTP expires in : 10
Testing OTP expires in : 9
Testing OTP expires in : 8
Testing OTP expires in : 7
Testing OTP expires in : 6
Testing OTP expires in : 5
Testing OTP expires in : 4
Testing OTP expires in : 4
Testing OTP expires in : 3
Testing OTP expires in : 2
Testing OTP expires in : 1


In [31]:
# ===== EXPIREAT =====
# Set expiration to specific timestamp
from time import time

future_timestamp = int(time()) + 60  # 60 seconds from now
r.set('expiring_key', 'value')
r.expireat('expiring_key', future_timestamp)

ttl = r.ttl('expiring_key')
print(f"Key expires in ~{ttl} seconds")

Key expires in ~51 seconds


## 3.2 Transactions

Execute multiple commands atomically (all-or-nothing).
Ensures data consistency and prevents race conditions.

In [32]:
# ===== BASIC TRANSACTION =====
# Use pipe() for transactions
r.set('account_balance', 1000)  # Initial balance

# Create a pipeline (transaction)
pipe = r.pipeline()

# Queue commands
pipe.get('account_balance')
pipe.decrby('account_balance', 100)  # Withdraw $100
pipe.get('account_balance')

# Execute all commands atomically
results = pipe.execute()
print(f"Transaction results:")
print(f"  Before: {results[0]}")
print(f"  Decrby result: {results[1]}")
print(f"  After: {results[2]}")

Transaction results:
  Before: 1000
  Decrby result: 900
  After: 900


In [33]:
# ===== TRANSACTION WITH ERROR HANDLING =====
def transfer_money(from_account, to_account, amount):
    """
    Transfer money between accounts atomically.
    Returns True if successful, False if insufficient funds.
    """
    # Setup accounts
    r.set(f'account:{from_account}', 1000)
    r.set(f'account:{to_account}', 500)
    
    try:
        pipe = r.pipeline()
        
        # Watch the from_account key for changes
        pipe.watch(f'account:{from_account}')
        
        # Check balance
        balance = r.get(f'account:{from_account}')
        if int(balance) < amount:
            pipe.unwatch()
            return False  # Insufficient funds
        
        # Proceed with transfer
        pipe.multi()
        pipe.decrby(f'account:{from_account}', amount)
        pipe.incrby(f'account:{to_account}', amount)
        pipe.execute()
        
        return True
    except redis.WatchError:
        print("Transaction aborted due to watched key changes")
        return False

# Test transfer
success = transfer_money('alice', 'bob', 200)
print(f"Transfer successful: {success}")
print(f"Alice's balance: {r.get('account:alice')}")
print(f"Bob's balance: {r.get('account:bob')}")

Transfer successful: True
Alice's balance: 800
Bob's balance: 700


## 3.3 Pub/Sub Messaging

Publish messages to channels and subscribe to receive them.
Great for real-time event notification systems.

In [34]:
# ===== BASIC PUB/SUB EXAMPLE =====
# Note: Pub/Sub requires separate connections for publisher and subscriber

# Publisher connection
r_pub = redis.Redis(host='localhost', port=6379, decode_responses=True)

# Subscriber connection
r_sub = redis.Redis(host='localhost', port=6379, decode_responses=True)

print("Pub/Sub example (for demonstration):")
print("""
# Subscriber listens to 'notifications' channel:
pubsub = r_sub.pubsub()
pubsub.subscribe('notifications')

for message in pubsub.listen():
    print(f"Received: {message['data']}")

# Publisher sends message:
r_pub.publish('notifications', 'Hello from publisher!')

# In a real scenario, you'd run subscriber in a separate thread/process
""")

Pub/Sub example (for demonstration):

# Subscriber listens to 'notifications' channel:
pubsub = r_sub.pubsub()
pubsub.subscribe('notifications')

for message in pubsub.listen():
    print(f"Received: {message['data']}")

# Publisher sends message:
r_pub.publish('notifications', 'Hello from publisher!')

# In a real scenario, you'd run subscriber in a separate thread/process



## 3.4 Pipelining

Send multiple commands to Redis without waiting for responses.
Significantly improves performance for bulk operations.

In [35]:
# ===== PIPELINE PERFORMANCE COMPARISON =====
import time

# Method 1: Without pipelining (slow)
r.delete('counter_slow')
start = time.time()
for i in range(100):
    r.incr('counter_slow')
time_without_pipe = time.time() - start
print(f"Without pipelining: {time_without_pipe:.4f} seconds")

# Method 2: With pipelining (fast)
r.delete('counter_fast')
start = time.time()
pipe = r.pipeline()
for i in range(100):
    pipe.incr('counter_fast')
pipe.execute()
time_with_pipe = time.time() - start
print(f"With pipelining: {time_with_pipe:.4f} seconds")
print(f"Speedup: {time_without_pipe / time_with_pipe:.1f}x faster")

Without pipelining: 0.0767 seconds
With pipelining: 0.0020 seconds
Speedup: 38.6x faster


In [36]:
# ===== BATCH OPERATIONS WITH PIPELINE =====
# Efficiently set multiple user profiles
users_data = [
    {'id': 1, 'name': 'Alice', 'score': 100},
    {'id': 2, 'name': 'Bob', 'score': 85},
    {'id': 3, 'name': 'Charlie', 'score': 95},
]

pipe = r.pipeline()
for user in users_data:
    key = f'user:{user["id"]}'
    pipe.hset(key, mapping={
        'name': user['name'],
        'score': user['score']
    })
    pipe.expire(key, 3600)  # 1 hour expiration

results = pipe.execute()
print(f"Batch operation completed. Commands executed: {len(results)}")
print(f"User profiles created: {len(users_data)}")

ResponseError: Command # 3 (HSET user:2 name Bob score 85) of pipeline caused error: WRONGTYPE Operation against a key holding the wrong kind of value

---
# Module 4: Design Patterns

## 4.1 Caching Patterns

Cache commonly accessed data to reduce database load.

In [ ]:
# ===== CACHE-ASIDE (Lazy Loading) PATTERN =====
def get_user_from_db(user_id):
    """
    Simulate database query.
    In reality, this would hit your database.
    """
    print(f"  [DB] Fetching user {user_id}...")
    return {
        'id': user_id,
        'name': f'User {user_id}',
        'email': f'user{user_id}@example.com'
    }

def get_user_cached(user_id, cache_ttl=3600):
    """
    Cache-aside pattern:
    1. Check cache
    2. If not found, fetch from DB
    3. Store in cache with TTL
    4. Return data
    """
    cache_key = f'user:{user_id}'
    
    # Step 1: Try cache
    cached_user = r.get(cache_key)
    if cached_user:
        print(f"  [CACHE HIT] Found user {user_id}")
        return json.loads(cached_user)
    
    # Step 2: Fetch from DB if not in cache
    user = get_user_from_db(user_id)
    
    # Step 3: Store in cache
    r.setex(cache_key, cache_ttl, json.dumps(user))
    
    return user
# First call - fetches from DB
print("First request:")
r.delete(f'user:1')  # Clear any existing key to avoid type conflicts
user1 = get_user_cached(1)
print(f"  Result: {user1['name']}\n")
# Second call - fetches from cache
print("Second request (same user):")
user1_again = get_user_cached(1)
print(f"  Result: {user1_again['name']}")

First request:
  [DB] Fetching user 1...
  Result: User 1

Second request (same user):
  [CACHE HIT] Found user 1
  Result: User 1


In [ ]:
# ===== WRITE-THROUGH CACHING PATTERN =====
def update_user_write_through(user_id, user_data):
    """
    Write-through pattern:
    1. Write to cache first
    2. Write to database
    3. Return success/failure
    
    Ensures cache and DB are always in sync.
    """
    cache_key = f'user:{user_id}'
    
    # Write to cache
    r.set(cache_key, json.dumps(user_data))
    
    # Write to database (simulated)
    print(f"  [DB] Writing user {user_id} to database...")
    # db.update_user(user_id, user_data)  # Your DB operation
    
    print(f"  [CACHE] User {user_id} cached")
    return True

# Test write-through
print("Updating user with write-through:")
update_user_write_through(2, {'id': 2, 'name': 'Updated User', 'email': 'updated@example.com'})
print(f"Cache value: {r.get('user:2')}")

Updating user with write-through:
  [DB] Writing user 2 to database...
  [CACHE] User 2 cached
Cache value: {"id": 2, "name": "Updated User", "email": "updated@example.com"}


## 4.2 Rate Limiting

Control the rate of incoming requests using Redis counters.

In [ ]:
# ===== SLIDING WINDOW RATE LIMITING =====
def is_request_allowed(user_id, max_requests=5, window_seconds=60):
    """
    Sliding window rate limiter.
    Allows max_requests per window_seconds for each user.
    
    Args:
        user_id: Unique user identifier
        max_requests: Maximum allowed requests
        window_seconds: Time window in seconds
    
    Returns:
        Tuple: (allowed: bool, remaining_requests: int)
    """
    key = f'ratelimit:{user_id}'
    current_count = r.incr(key)
    
    # Set TTL on first request
    if current_count == 1:
        r.expire(key, window_seconds)
    
    remaining = max(0, max_requests - current_count)
    allowed = current_count <= max_requests
    
    return allowed, remaining

# Simulate user requests
print("Rate Limiting Demo (5 requests per 60 seconds):\n")
for request_num in range(8):
    allowed, remaining = is_request_allowed('user:123')
    status = '✓ ALLOWED' if allowed else '✗ BLOCKED'
    print(f"Request {request_num + 1}: {status} ({remaining} remaining)")

Rate Limiting Demo (5 requests per 60 seconds):

Request 1: ✗ BLOCKED (0 remaining)
Request 2: ✗ BLOCKED (0 remaining)
Request 3: ✗ BLOCKED (0 remaining)
Request 4: ✗ BLOCKED (0 remaining)
Request 5: ✗ BLOCKED (0 remaining)
Request 6: ✗ BLOCKED (0 remaining)
Request 7: ✗ BLOCKED (0 remaining)
Request 8: ✗ BLOCKED (0 remaining)


## 4.3 Session Management

Store and manage user sessions efficiently.

In [ ]:
# ===== SESSION STORAGE =====
def create_session(user_id, session_duration=3600):
    """
    Create a user session.
    
    Args:
        user_id: User ID
        session_duration: Session lifetime in seconds
    
    Returns:
        Session token
    """
    import uuid
    
    session_token = str(uuid.uuid4())
    session_key = f'session:{session_token}'
    
    # Store session data
    session_data = {
        'user_id': user_id,
        'created_at': datetime.now().isoformat(),
        'ip_address': '192.168.1.100',
        'user_agent': 'Mozilla/5.0'
    }
    
    r.setex(
        session_key,
        session_duration,
        json.dumps(session_data)
    )
    
    # Also store reverse mapping for quick lookup
    r.setex(
        f'user:{user_id}:session',
        session_duration,
        session_token
    )
    
    return session_token

def get_session(session_token):
    """Retrieve session data."""
    session_data = r.get(f'session:{session_token}')
    if session_data:
        return json.loads(session_data)
    return None

def destroy_session(session_token):
    """Logout - destroy session."""
    session_data = get_session(session_token)
    if session_data:
        user_id = session_data['user_id']
        r.delete(f'session:{session_token}')
        r.delete(f'user:{user_id}:session')
        return True
    return False

# Test session management
print("Session Management Demo:\n")
token = create_session('alice', session_duration=3600)
print(f"Created session: {token[:12]}...\n")

session = get_session(token)
print(f"Retrieved session: {json.dumps(session, indent=2)}\n")

destroyed = destroy_session(token)
print(f"Session destroyed: {destroyed}")
print(f"Session after logout: {get_session(token)}")

Session Management Demo:

Created session: 2e5bf8fc-2c5...

Retrieved session: {
  "user_id": "alice",
  "created_at": "2026-05-29T12:10:35.564377",
  "ip_address": "192.168.1.100",
  "user_agent": "Mozilla/5.0"
}

Session destroyed: True
Session after logout: None


## 4.4 Distributed Locks

Implement mutual exclusion for concurrent operations.

In [ ]:
# ===== SIMPLE DISTRIBUTED LOCK =====
import uuid
import time

class RedisLock:
    """
    Simple Redis-based distributed lock.
    
    Usage:
        lock = RedisLock(redis_conn, 'resource_name')
        if lock.acquire(timeout=5):
            try:
                # Critical section
                pass
            finally:
                lock.release()
    """
    
    def __init__(self, redis_conn, lock_name):
        self.redis = redis_conn
        self.lock_name = f'lock:{lock_name}'
        self.token = None  # Unique token for this lock holder
    
    def acquire(self, timeout=10, blocking=True):
        """
        Try to acquire the lock.
        
        Args:
            timeout: Lock TTL (prevents deadlock)
            blocking: Wait for lock or fail immediately
        
        Returns:
            True if acquired, False otherwise
        """
        self.token = str(uuid.uuid4())
        end_time = time.time() + timeout
        
        while True:
            # Try to set lock (NX = only if not exists)
            acquired = self.redis.set(
                self.lock_name,
                self.token,
                ex=10,  # Auto-expire after 10 seconds
                nx=True  # Only if not exists
            )
            
            if acquired:
                return True
            
            if not blocking or time.time() > end_time:
                return False
            
            time.sleep(0.01)  # Avoid busy-waiting
    
    def release(self):
        """
        Release the lock (only if we own it).
        """
        # Check if we still own the lock
        current_token = self.redis.get(self.lock_name)
        
        if current_token == self.token:
            self.redis.delete(self.lock_name)
            return True
        
        return False

# Test distributed lock
print("Distributed Lock Demo:\n")
lock = RedisLock(r, 'critical_resource')

if lock.acquire(blocking=True):
    print("✓ Lock acquired")
    print(f"  Token: {lock.token[:12]}...")
    print("  Performing critical operation...")
    time.sleep(1)
    lock.release()
    print("✓ Lock released")
else:
    print("✗ Failed to acquire lock")

Distributed Lock Demo:

✓ Lock acquired
  Token: a0573b55-6bb...
  Performing critical operation...
✓ Lock released


---
# Module 5: Performance & Optimization

## 5.1 Key Design Strategies

Design efficient keys for performance and maintainability.

In [ ]:
# ===== KEY NAMING CONVENTIONS =====
print("Redis Key Naming Best Practices:\n")

# Good: Clear, hierarchical naming
examples_good = [
    'user:1000:profile',          # User profile data
    'user:1000:sessions:abc123',  # User session
    'post:5000',                   # Post data
    'like:post:5000:user:1000',   # Like relationship
    'cache:user:profile:1000',    # Cache namespace
    'ratelimit:api:user:1000',    # Rate limit counter
]

print("Good key names:")
for key in examples_good:
    print(f"  ✓ {key}")

print("\nBad key names:")
examples_bad = [
    'user1000',                    # Not hierarchical
    'data',                        # Too vague
    'u1000p',                      # Unclear abbreviations
    'user-1000-profile',           # Inconsistent separators
]

for key in examples_bad:
    print(f"  ✗ {key}")

print("\nKey Design Principles:")
principles = [
    "Use colons (:) as separators for hierarchy",
    "Be verbose - keys are not stored; memory is cheap for clarity",
    "Use namespaces to organize related keys",
    "Include type information (e.g., cache:, ratelimit:)",
    "Keep keys consistent across your application"
]
for i, principle in enumerate(principles, 1):
    print(f"  {i}. {principle}")

Redis Key Naming Best Practices:

Good key names:
  ✓ user:1000:profile
  ✓ user:1000:sessions:abc123
  ✓ post:5000
  ✓ like:post:5000:user:1000
  ✓ cache:user:profile:1000
  ✓ ratelimit:api:user:1000

Bad key names:
  ✗ user1000
  ✗ data
  ✗ u1000p
  ✗ user-1000-profile

Key Design Principles:
  1. Use colons (:) as separators for hierarchy
  2. Be verbose - keys are not stored; memory is cheap for clarity
  3. Use namespaces to organize related keys
  4. Include type information (e.g., cache:, ratelimit:)
  5. Keep keys consistent across your application


## 5.2 Memory Management

Optimize Redis memory usage.

In [ ]:
# ===== MEMORY ANALYSIS =====
def analyze_memory():
    """
    Analyze Redis memory usage.
    """
    info = r.info('memory')
    
    print("Memory Statistics:\n")
    print(f"Used Memory: {info.get('used_memory_human')}")
    print(f"Peak Memory: {info.get('used_memory_peak_human')}")
    print(f"Memory Fragmentation: {info.get('mem_fragmentation_ratio'):.2f}")
    print(f"Memory Policy: {info.get('maxmemory_policy')}")

analyze_memory()

Memory Statistics:

Used Memory: 2.77M
Peak Memory: 2.77M
Memory Fragmentation: 8.31
Memory Policy: noeviction


In [ ]:
# ===== EVICTION POLICIES =====
print("\nRedis Eviction Policies (when maxmemory is reached):\n")

policies = {
    'noeviction': 'Don\'t evict; return error when memory full',
    'allkeys-lru': 'Remove any key using LRU (Least Recently Used)',
    'allkeys-lfu': 'Remove any key using LFU (Least Frequently Used)',
    'volatile-lru': 'Remove expiring keys using LRU',
    'volatile-lfu': 'Remove expiring keys using LFU',
    'allkeys-random': 'Remove random key',
    'volatile-random': 'Remove random expiring key',
    'volatile-ttl': 'Remove key with shortest TTL',
}

for policy, description in policies.items():
    print(f"• {policy}")
    print(f"  → {description}\n")


Redis Eviction Policies (when maxmemory is reached):

• noeviction
  → Don't evict; return error when memory full

• allkeys-lru
  → Remove any key using LRU (Least Recently Used)

• allkeys-lfu
  → Remove any key using LFU (Least Frequently Used)

• volatile-lru
  → Remove expiring keys using LRU

• volatile-lfu
  → Remove expiring keys using LFU

• allkeys-random
  → Remove random key

• volatile-random
  → Remove random expiring key

• volatile-ttl
  → Remove key with shortest TTL



## 5.3 Persistence

Ensure data survives server restarts.

In [ ]:
# ===== PERSISTENCE OPTIONS =====
print("Redis Persistence Options:\n")

print("1. RDB (Redis Database Dump)")
print("   - Snapshot-based persistence")
print("   - Smaller file size")
print("   - Faster recovery")
print("   - May lose recent data on crash\n")

print("2. AOF (Append-Only File)")
print("   - Command log-based persistence")
print("   - No data loss (fsync every command)")
print("   - Larger file size")
print("   - Slower performance\n")

print("3. RDB + AOF (Hybrid)")
print("   - Best of both worlds")
print("   - Recommended for production\n")

print("Configuration (redis.conf):")
config_example = """
# RDB Snapshots
save 900 1          # After 900 sec if 1+ key changed
save 300 10         # After 300 sec if 10+ keys changed
save 60 10000       # After 60 sec if 10000+ keys changed

# AOF
appendonly yes
appendfsync everysec  # fsync every second (default)
"""
print(config_example)

Redis Persistence Options:

1. RDB (Redis Database Dump)
   - Snapshot-based persistence
   - Smaller file size
   - Faster recovery
   - May lose recent data on crash

2. AOF (Append-Only File)
   - Command log-based persistence
   - No data loss (fsync every command)
   - Larger file size
   - Slower performance

3. RDB + AOF (Hybrid)
   - Best of both worlds
   - Recommended for production

Configuration (redis.conf):

# RDB Snapshots
save 900 1          # After 900 sec if 1+ key changed
save 300 10         # After 300 sec if 10+ keys changed
save 60 10000       # After 60 sec if 10000+ keys changed

# AOF
appendonly yes
appendfsync everysec  # fsync every second (default)



## 5.4 Monitoring

Monitor Redis performance.

In [ ]:
# ===== REDIS STATISTICS =====
def print_redis_stats():
    """
    Display comprehensive Redis statistics.
    """
    info = r.info()
    
    print("=== REDIS SERVER STATS ===")
    print(f"Version: {info['redis_version']}")
    print(f"Process ID: {info['process_id']}")
    print(f"Uptime: {info['uptime_in_seconds']} seconds\n")
    
    print("=== MEMORY ===")
    print(f"Used: {info['used_memory_human']}")
    print(f"Peak: {info['used_memory_peak_human']}")
    print(f"Fragmentation: {info['mem_fragmentation_ratio']}\n")
    
    print("=== STATS ===")
    print(f"Total Connections: {info['total_connections_received']}")
    print(f"Current Clients: {info['connected_clients']}")
    print(f"Commands Processed: {info['total_commands_processed']}")
    print(f"Keyspace Hits: {info.get('keyspace_hits', 0)}")
    print(f"Keyspace Misses: {info.get('keyspace_misses', 0)}")
    
    # Calculate hit rate
    hits = info.get('keyspace_hits', 0)
    misses = info.get('keyspace_misses', 0)
    if hits + misses > 0:
        hit_rate = hits / (hits + misses) * 100
        print(f"Hit Rate: {hit_rate:.2f}%")

print_redis_stats()

=== REDIS SERVER STATS ===
Version: 8.8.0
Process ID: 1
Uptime: 5097 seconds

=== MEMORY ===
Used: 2.77M
Peak: 2.79M
Fragmentation: 8.31

=== STATS ===
Total Connections: 22
Current Clients: 5
Commands Processed: 997
Keyspace Hits: 217
Keyspace Misses: 10
Hit Rate: 95.59%


In [ ]:
# ===== KEYSPACE ANALYSIS =====
print("\n=== KEYSPACE ANALYSIS ===")
keyspace_info = r.info('keyspace')

for db, stats in keyspace_info.items():
    if db.startswith('db'):
        print(f"{db}: {stats}")


=== KEYSPACE ANALYSIS ===
db0: {'keys': 22, 'expires': 2, 'avg_ttl': 1362311, 'subexpiry': 0}


---
# Module 6: FastAPI Integration

## 6.1 Setting up Redis with FastAPI

Integrate Redis as a cache and session store in FastAPI applications.

### Installation & Setup

In [ ]:
# Install required packages:
# pip install fastapi uvicorn redis aioredis

print("✓ Required packages installed")
print("  - fastapi: Web framework")
print("  - uvicorn: ASGI server")
print("  - redis: Redis client")
print("  - aioredis: Async Redis client")

✓ Required packages installed
  - fastapi: Web framework
  - uvicorn: ASGI server
  - redis: Redis client
  - aioredis: Async Redis client


### Basic FastAPI + Redis Application

In [ ]:
# Create file: fastapi_redis_app.py

fastapi_code = '''
from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel
import redis
import json
from typing import Optional
from datetime import datetime

# Create FastAPI app
app = FastAPI(title="Redis + FastAPI Demo")

# Redis connection pool
redis_client = redis.Redis(
    host='localhost',
    port=6379,
    db=0,
    decode_responses=True
)

# ===== MODELS =====
class User(BaseModel):
    """User data model."""
    id: int
    name: str
    email: str
    age: int

class Post(BaseModel):
    """Blog post model."""
    id: int
    title: str
    content: str
    author_id: int

# ===== DEPENDENCY =====
def get_redis():
    """Dependency to get Redis connection."""
    return redis_client

# ===== ENDPOINTS =====

@app.get("/health")
def health_check(redis: redis.Redis = Depends(get_redis)):
    """Check if Redis is connected."""
    try:
        redis.ping()
        return {"status": "healthy", "redis": "connected"}
    except:
        return {"status": "unhealthy", "redis": "disconnected"}

@app.post("/users/")
def create_user(user: User, redis: redis.Redis = Depends(get_redis)):
    """
    Create a new user.
    Stores in Redis for fast access.
    """
    user_key = f"user:{user.id}"
    
    # Check if user already exists
    if redis.exists(user_key):
        raise HTTPException(status_code=400, detail="User already exists")
    
    # Store user data (with 1-hour expiration)
    redis.setex(
        user_key,
        3600,
        json.dumps(user.dict())
    )
    
    return {"message": "User created", "user": user}

@app.get("/users/{user_id}")
def get_user(user_id: int, redis: redis.Redis = Depends(get_redis)):
    """
    Retrieve user by ID.
    Uses cache-aside pattern.
    """
    user_key = f"user:{user_id}"
    
    # Try cache first
    cached_user = redis.get(user_key)
    if cached_user:
        return {"source": "cache", "user": json.loads(cached_user)}
    
    # If not in cache, simulate fetching from database
    # db_user = database.get_user(user_id)
    # if not db_user:
    #     raise HTTPException(status_code=404, detail="User not found")
    # 
    # Cache the result
    # redis.setex(user_key, 3600, json.dumps(db_user.dict()))
    # return {"source": "database", "user": db_user}
    
    raise HTTPException(status_code=404, detail="User not found")

@app.delete("/users/{user_id}")
def delete_user(user_id: int, redis: redis.Redis = Depends(get_redis)):
    """
    Delete user.
    Removes from both cache and database.
    """
    user_key = f"user:{user_id}"
    
    # Delete from cache
    deleted = redis.delete(user_key)
    
    if deleted:
        return {"message": "User deleted"}
    else:
        raise HTTPException(status_code=404, detail="User not found")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("FastAPI + Redis Application Code:")
print("="*60)
print(fastapi_code)
print("="*60)
print("\nSave this as 'fastapi_redis_app.py'")
print("Run with: uvicorn fastapi_redis_app:app --reload")

FastAPI + Redis Application Code:

from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel
import redis
import json
from typing import Optional
from datetime import datetime

# Create FastAPI app
app = FastAPI(title="Redis + FastAPI Demo")

# Redis connection pool
redis_client = redis.Redis(
    host='localhost',
    port=6379,
    db=0,
    decode_responses=True
)

# ===== MODELS =====
class User(BaseModel):
    """User data model."""
    id: int
    name: str
    email: str
    age: int

class Post(BaseModel):
    """Blog post model."""
    id: int
    title: str
    content: str
    author_id: int

# ===== DEPENDENCY =====
def get_redis():
    """Dependency to get Redis connection."""
    return redis_client

# ===== ENDPOINTS =====

@app.get("/health")
def health_check(redis: redis.Redis = Depends(get_redis)):
    """Check if Redis is connected."""
    try:
        redis.ping()
        return {"status": "healthy", "redis": "connected"}
    except:
       

## 6.2 Caching API Responses

Use Redis to cache expensive API responses.

In [ ]:
# Response caching decorator

response_cache_code = '''
import json
from functools import wraps
from typing import Callable
import redis
from hashlib import md5

class ResponseCache:
    """
    Decorator for caching FastAPI endpoint responses.
    
    Usage:
        cache = ResponseCache(redis_client, ttl=3600)
        
        @app.get("/expensive-endpoint")
        @cache.cached()
        def expensive_endpoint():
            return {"data": "...expensive computation..."}
    """
    
    def __init__(self, redis_client: redis.Redis, ttl: int = 3600):
        self.redis = redis_client
        self.ttl = ttl
    
    def _get_cache_key(self, func_name: str, args: tuple, kwargs: dict) -> str:
        """
        Generate cache key from function name and arguments.
        """
        key_data = f"{func_name}:{str(args)}:{str(kwargs)}"
        key_hash = md5(key_data.encode()).hexdigest()
        return f"cache:response:{key_hash}"
    
    def cached(self, ttl: int = None):
        """
        Decorator to cache function response.
        """
        def decorator(func: Callable):
            @wraps(func)
            def wrapper(*args, **kwargs):
                cache_key = self._get_cache_key(func.__name__, args, kwargs)
                
                # Try cache
                cached_response = self.redis.get(cache_key)
                if cached_response:
                    return json.loads(cached_response)
                
                # Compute response
                response = func(*args, **kwargs)
                
                # Cache response
                cache_ttl = ttl or self.ttl
                self.redis.setex(
                    cache_key,
                    cache_ttl,
                    json.dumps(response)
                )
                
                return response
            
            return wrapper
        return decorator

# Example usage in FastAPI:
"""
from fastapi import FastAPI

app = FastAPI()
redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
cache = ResponseCache(redis_client, ttl=3600)

@app.get("/products/")
@cache.cached()
def get_products():
    # Expensive computation
    return {
        "products": [
            {"id": 1, "name": "Product 1"},
            {"id": 2, "name": "Product 2"},
        ]
    }
"""
'''

print("Response Cache Decorator:")
print("="*60)
print(response_cache_code)
print("="*60)

Response Cache Decorator:

import json
from functools import wraps
from typing import Callable
import redis
from hashlib import md5

class ResponseCache:
    """
    Decorator for caching FastAPI endpoint responses.

    Usage:
        cache = ResponseCache(redis_client, ttl=3600)

        @app.get("/expensive-endpoint")
        @cache.cached()
        def expensive_endpoint():
            return {"data": "...expensive computation..."}
    """

    def __init__(self, redis_client: redis.Redis, ttl: int = 3600):
        self.redis = redis_client
        self.ttl = ttl

    def _get_cache_key(self, func_name: str, args: tuple, kwargs: dict) -> str:
        """
        Generate cache key from function name and arguments.
        """
        key_data = f"{func_name}:{str(args)}:{str(kwargs)}"
        key_hash = md5(key_data.encode()).hexdigest()
        return f"cache:response:{key_hash}"

    def cached(self, ttl: int = None):
        """
        Decorator to cache function response.
    

## 6.3 Rate Limiting Middleware

Implement rate limiting as FastAPI middleware.

In [ ]:
# Rate limiting middleware

rate_limit_middleware = '''
from fastapi import FastAPI, Request, HTTPException
from starlette.middleware.base import BaseHTTPMiddleware
import redis
from datetime import datetime, timedelta

class RateLimitMiddleware(BaseHTTPMiddleware):
    """
    Rate limiting middleware for FastAPI.
    
    Limits requests per user/IP address.
    """
    
    def __init__(self, app, redis_client: redis.Redis, requests_per_minute: int = 60):
        super().__init__(app)
        self.redis = redis_client
        self.requests_per_minute = requests_per_minute
    
    async def dispatch(self, request: Request, call_next):
        # Get client IP
        client_ip = request.client.host
        
        # Get rate limit key
        rate_limit_key = f"ratelimit:{client_ip}:{request.url.path}"
        
        # Increment counter
        current_count = self.redis.incr(rate_limit_key)
        
        # Set expiration on first request
        if current_count == 1:
            self.redis.expire(rate_limit_key, 60)  # 1 minute window
        
        # Check if exceeded limit
        if current_count > self.requests_per_minute:
            raise HTTPException(
                status_code=429,
                detail="Rate limit exceeded"
            )
        
        # Add rate limit headers
        response = await call_next(request)
        response.headers["X-RateLimit-Limit"] = str(self.requests_per_minute)
        response.headers["X-RateLimit-Remaining"] = str(
            self.requests_per_minute - current_count
        )
        
        return response

# Usage:
"""
from fastapi import FastAPI
import redis

app = FastAPI()
redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

# Add middleware
app.add_middleware(
    RateLimitMiddleware,
    redis_client=redis_client,
    requests_per_minute=100
)
"""
'''

print("Rate Limiting Middleware:")
print("="*60)
print(rate_limit_middleware)
print("="*60)

Rate Limiting Middleware:

from fastapi import FastAPI, Request, HTTPException
from starlette.middleware.base import BaseHTTPMiddleware
import redis
from datetime import datetime, timedelta

class RateLimitMiddleware(BaseHTTPMiddleware):
    """
    Rate limiting middleware for FastAPI.

    Limits requests per user/IP address.
    """

    def __init__(self, app, redis_client: redis.Redis, requests_per_minute: int = 60):
        super().__init__(app)
        self.redis = redis_client
        self.requests_per_minute = requests_per_minute

    async def dispatch(self, request: Request, call_next):
        # Get client IP
        client_ip = request.client.host

        # Get rate limit key
        rate_limit_key = f"ratelimit:{client_ip}:{request.url.path}"

        # Increment counter
        current_count = self.redis.incr(rate_limit_key)

        # Set expiration on first request
        if current_count == 1:
            self.redis.expire(rate_limit_key, 60)  # 1 minute window

   

## 6.4 Real-time Features with Pub/Sub

Implement WebSocket support for real-time updates using Redis Pub/Sub.

In [ ]:
# Real-time WebSocket with Pub/Sub

websocket_pubsub = '''
from fastapi import FastAPI, WebSocket
import redis
import json
import asyncio
from threading import Thread

app = FastAPI()

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
pubsub = redis_client.pubsub()

class ConnectionManager:
    """
    Manage WebSocket connections for real-time updates.
    """
    
    def __init__(self):
        self.active_connections: list[WebSocket] = []
    
    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)
    
    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)
    
    async def broadcast(self, message: dict):
        """
        Send message to all connected clients.
        """
        for connection in self.active_connections:
            try:
                await connection.send_json(message)
            except Exception as e:
                print(f"Error sending message: {e}")

manager = ConnectionManager()

@app.websocket("/ws/notifications")
async def websocket_endpoint(websocket: WebSocket):
    """
    WebSocket endpoint for real-time notifications.
    """
    await manager.connect(websocket)
    
    try:
        while True:
            # Receive from client
            data = await websocket.receive_text()
            
            # Publish to Redis channel
            message = json.loads(data)
            redis_client.publish('notifications', json.dumps(message))
            
            # Broadcast to all connected clients
            await manager.broadcast({"event": "new_notification", "data": message})
    
    except Exception as e:
        print(f"WebSocket error: {e}")
    
    finally:
        manager.disconnect(websocket)

def redis_listener():
    """
    Listen for Redis Pub/Sub messages in background thread.
    """
    pubsub.subscribe('notifications')
    
    for message in pubsub.listen():
        if message['type'] == 'message':
            print(f"Notification: {message['data']}")

# Start listener thread
listener_thread = Thread(target=redis_listener, daemon=True)
listener_thread.start()

@app.post("/send-notification/")
def send_notification(message: dict):
    """
    Send notification via Redis Pub/Sub.
    All connected WebSocket clients receive it.
    """
    redis_client.publish('notifications', json.dumps(message))
    return {"status": "notification sent"}
'''

print("Real-time WebSocket with Pub/Sub:")
print("="*60)
print(websocket_pubsub)
print("="*60)

Real-time WebSocket with Pub/Sub:

from fastapi import FastAPI, WebSocket
import redis
import json
import asyncio
from threading import Thread

app = FastAPI()

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
pubsub = redis_client.pubsub()

class ConnectionManager:
    """
    Manage WebSocket connections for real-time updates.
    """

    def __init__(self):
        self.active_connections: list[WebSocket] = []

    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)

    async def broadcast(self, message: dict):
        """
        Send message to all connected clients.
        """
        for connection in self.active_connections:
            try:
                await connection.send_json(message)
            except Exception as e:
                print(f"Error sending message

## 6.5 Complete FastAPI Example with All Features

A comprehensive example combining caching, rate limiting, sessions, and Pub/Sub.

In [ ]:
complete_fastapi_example = '''
# File: complete_app.py

from fastapi import FastAPI, HTTPException, Request, WebSocket, Depends
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware
from pydantic import BaseModel
import redis
import json
import uuid
from datetime import datetime
from typing import Optional
from functools import wraps
import asyncio
from threading import Thread

# ===== INITIALIZATION =====
app = FastAPI(title="Advanced Redis + FastAPI")

redis_client = redis.Redis(
    host='localhost',
    port=6379,
    db=0,
    decode_responses=True
)

# ===== MODELS =====
class User(BaseModel):
    username: str
    email: str
    age: int

class LoginRequest(BaseModel):
    username: str
    password: str

class CacheConfig(BaseModel):
    ttl: int = 3600
    namespace: str = "cache"

# ===== RATE LIMIT MIDDLEWARE =====
class RateLimitMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        # Skip rate limiting for health checks
        if request.url.path == "/health":
            return await call_next(request)
        
        client_ip = request.client.host
        rate_limit_key = f"ratelimit:{client_ip}:{request.url.path}"
        
        current = redis_client.incr(rate_limit_key)
        if current == 1:
            redis_client.expire(rate_limit_key, 60)
        
        if current > 100:  # 100 requests per minute
            return JSONResponse(
                status_code=429,
                content={"detail": "Rate limit exceeded"},
                headers={
                    "X-RateLimit-Limit": "100",
                    "X-RateLimit-Remaining": "0"
                }
            )
        
        response = await call_next(request)
        response.headers["X-RateLimit-Limit"] = "100"
        response.headers["X-RateLimit-Remaining"] = str(100 - current)
        return response

app.add_middleware(RateLimitMiddleware)

# ===== DEPENDENCIES =====
def get_redis():
    return redis_client

async def get_current_user(request: Request, redis: redis.Redis = Depends(get_redis)):
    """
    Get current authenticated user from session.
    """
    token = request.headers.get("Authorization")
    if not token:
        raise HTTPException(status_code=401, detail="Not authenticated")
    
    session_data = redis.get(f"session:{token}")
    if not session_data:
        raise HTTPException(status_code=401, detail="Invalid session")
    
    return json.loads(session_data)

# ===== ENDPOINTS =====

@app.get("/health")
def health_check(redis: redis.Redis = Depends(get_redis)):
    """Health check endpoint."""
    try:
        redis.ping()
        return {"status": "ok", "timestamp": datetime.now().isoformat()}
    except:
        raise HTTPException(status_code=503, detail="Redis unavailable")

@app.post("/login")
def login(credentials: LoginRequest, redis: redis.Redis = Depends(get_redis)):
    """
    Login endpoint.
    Creates session in Redis.
    """
    # Simulate authentication (in real app, verify against DB)
    if credentials.username and credentials.password:
        session_token = str(uuid.uuid4())
        session_data = {
            "username": credentials.username,
            "created_at": datetime.now().isoformat(),
            "permissions": ["read", "write"]
        }
        
        # Store session with 1-hour expiration
        redis.setex(
            f"session:{session_token}",
            3600,
            json.dumps(session_data)
        )
        
        return {
            "access_token": session_token,
            "token_type": "bearer",
            "user": session_data
        }
    
    raise HTTPException(status_code=401, detail="Invalid credentials")

@app.post("/logout")
def logout(request: Request, redis: redis.Redis = Depends(get_redis)):
    """
    Logout endpoint.
    Destroys session.
    """
    token = request.headers.get("Authorization")
    if token:
        redis.delete(f"session:{token}")
    
    return {"message": "Logged out successfully"}

@app.get("/users/{user_id}")
def get_user(
    user_id: int,
    current_user: dict = Depends(get_current_user),
    redis: redis.Redis = Depends(get_redis)
):
    """
    Get user with caching.
    Cache-aside pattern.
    """
    cache_key = f"user:{user_id}"
    
    # Try cache
    cached = redis.get(cache_key)
    if cached:
        return {"source": "cache", "user": json.loads(cached)}
    
    # Simulate DB fetch
    user_data = {
        "id": user_id,
        "username": f"user_{user_id}",
        "email": f"user{user_id}@example.com"
    }
    
    # Cache result
    redis.setex(cache_key, 3600, json.dumps(user_data))
    
    return {"source": "database", "user": user_data}

@app.post("/cache-data")
def cache_data(
    key: str,
    value: dict,
    config: CacheConfig = CacheConfig(),
    current_user: dict = Depends(get_current_user),
    redis: redis.Redis = Depends(get_redis)
):
    """
    Manually cache data.
    """
    cache_key = f"{config.namespace}:{key}"
    redis.setex(cache_key, config.ttl, json.dumps(value))
    
    return {
        "message": "Data cached",
        "key": cache_key,
        "ttl": config.ttl
    }

@app.get("/cache-data/{key}")
def get_cached_data(
    key: str,
    current_user: dict = Depends(get_current_user),
    redis: redis.Redis = Depends(get_redis)
):
    """
    Retrieve cached data.
    """
    cache_key = f"cache:{key}"
    data = redis.get(cache_key)
    
    if data:
        return {"data": json.loads(data)}
    
    raise HTTPException(status_code=404, detail="Cache key not found")

@app.post("/publish-message")
def publish_message(
    channel: str,
    message: dict,
    current_user: dict = Depends(get_current_user),
    redis: redis.Redis = Depends(get_redis)
):
    """
    Publish message to Redis channel.
    For Pub/Sub messaging.
    """
    message_with_meta = {
        "author": current_user["username"],
        "content": message,
        "timestamp": datetime.now().isoformat()
    }
    
    subscribers = redis.publish(channel, json.dumps(message_with_meta))
    
    return {
        "message": "Message published",
        "channel": channel,
        "subscribers": subscribers
    }

@app.websocket("/ws/{client_id}")
async def websocket_endpoint(
    websocket: WebSocket,
    client_id: str,
    redis: redis.Redis = Depends(get_redis)
):
    """
    WebSocket endpoint for real-time updates.
    """
    await websocket.accept()
    
    # Register client
    redis.sadd("active_clients", client_id)
    
    try:
        while True:
            data = await websocket.receive_text()
            
            # Broadcast to all clients via Redis
            redis.publish("broadcasts", json.dumps({
                "client": client_id,
                "message": data,
                "timestamp": datetime.now().isoformat()
            }))
            
            # Echo back to sender
            await websocket.send_text(f"Echo: {data}")
    
    except Exception as e:
        print(f"WebSocket error: {e}")
    
    finally:
        redis.srem("active_clients", client_id)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("Complete FastAPI Application with Redis:")
print("="*60)
print(complete_fastapi_example)
print("="*60)
print("\nTo run this application:")
print("1. Save as 'complete_app.py'")
print("2. Install dependencies: pip install fastapi uvicorn redis")
print("3. Ensure Redis is running: redis-server")
print("4. Run: uvicorn complete_app:app --reload")
print("5. Access: http://localhost:8000/docs")

Complete FastAPI Application with Redis:

# File: complete_app.py

from fastapi import FastAPI, HTTPException, Request, WebSocket, Depends
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware
from pydantic import BaseModel
import redis
import json
import uuid
from datetime import datetime
from typing import Optional
from functools import wraps
import asyncio
from threading import Thread

# ===== INITIALIZATION =====
app = FastAPI(title="Advanced Redis + FastAPI")

redis_client = redis.Redis(
    host='localhost',
    port=6379,
    db=0,
    decode_responses=True
)

# ===== MODELS =====
class User(BaseModel):
    username: str
    email: str
    age: int

class LoginRequest(BaseModel):
    username: str
    password: str

class CacheConfig(BaseModel):
    ttl: int = 3600
    namespace: str = "cache"

# ===== RATE LIMIT MIDDLEWARE =====
class RateLimitMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):

---
# Summary & Best Practices

## Key Takeaways

### Redis Strengths
✓ **In-memory speed**: Sub-millisecond response times
✓ **Data structures**: Strings, Lists, Sets, Hashes, Sorted Sets
✓ **Atomic operations**: Built-in incrementers, transactions
✓ **Expiration**: Automatic key cleanup
✓ **Pub/Sub**: Real-time messaging
✓ **Persistence**: RDB snapshots and AOF logs

### Best Practices
1. **Use connection pooling** for efficiency
2. **Set appropriate TTLs** to prevent memory bloat
3. **Design keys hierarchically** (e.g., `user:1000:profile`)
4. **Use pipelining** for batch operations
5. **Monitor memory** and set eviction policies
6. **Implement rate limiting** for API protection
7. **Cache strategically** (not everything)
8. **Use Lua scripts** for complex atomic operations
9. **Handle failures gracefully** (treat as cache miss)
10. **Test under load** before production deployment

### When to Use Redis
- **Caching**: Reduce database load
- **Sessions**: Fast session storage
- **Rate limiting**: API throttling
- **Leaderboards**: Ranked data
- **Real-time features**: Pub/Sub messaging
- **Task queues**: Job processing
- **Counters**: Analytics and metrics

### When NOT to Use Redis
- Complex queries (use database)
- Persistent data (consider DB backup)
- Large datasets (memory constraints)
- Graph/relational data (use graph/relational DB)

## Useful Commands Reference

In [ ]:
# Quick reference of important Redis commands

reference = """
# STRING COMMANDS
SET key value [EX seconds]      # Set value with optional expiration
GET key                          # Get value
MSET key1 val1 key2 val2        # Set multiple key-value pairs
MGET key1 key2                   # Get multiple values
INCR key                         # Increment by 1
DECR key                         # Decrement by 1
APPEND key value                 # Append to string

# LIST COMMANDS
RPUSH key value                  # Push to right
LPUSH key value                  # Push to left
RPOP key                         # Pop from right
LPOP key                         # Pop from left
LLEN key                         # List length
LRANGE key start end             # Get range
LTRIM key start end              # Keep only range

# SET COMMANDS
SADD key member                  # Add member
REM key member                   # Remove member
SMEMBERS key                     # Get all members
SISMEMBER key member            # Check membership
SINTER key1 key2                # Intersection
SUNION key1 key2                # Union
SDIFF key1 key2                 # Difference

# HASH COMMANDS
HSET key field value            # Set hash field
HGET key field                  # Get hash field
HGETALL key                     # Get entire hash
HKEYS key                       # Get all field names
HVALS key                       # Get all values
HDEL key field                  # Delete field
HINCRBY key field increment     # Increment field

# SORTED SET COMMANDS
ZADD key score member           # Add member with score
ZRANGE key start end            # Get by index
ZREVRANGE key start end         # Get reversed
ZSCORE key member               # Get score
ZRANK key member                # Get rank
ZREVRANK key member             # Get reverse rank
ZCOUNT key min max              # Count in range

# KEY COMMANDS
DEL key                         # Delete key
EXISTS key                      # Check if exists
EXPIRE key seconds              # Set expiration
TTL key                         # Get time to live
PERSIST key                     # Remove expiration
KEYS pattern                    # Find keys (careful!)
DBSIZE                          # Total keys
FLUSHDB                         # Delete all keys
FLUSHALL                        # Delete all databases

# TRANSACTION COMMANDS
MULTI                           # Start transaction
EXEC                            # Execute transaction
DISCARD                         # Discard transaction
WATCH key                       # Watch key for changes
UNWATCH                         # Stop watching

# PUB/SUB COMMANDS
PUBLISH channel message         # Publish message
SUBSCRIBE channel              # Subscribe to channel
UNSUBSCRIBE channel            # Unsubscribe
PSUBSCRIBE pattern             # Pattern subscribe

# SCRIPT COMMANDS
EVAL script numkeys key [key]  # Execute Lua script
SCRIPT LOAD script             # Load script
SCRIPT EXISTS sha1             # Check if loaded

# SERVER COMMANDS
PING                            # Test connection
INFO                            # Server info
CONFIG GET parameter           # Get configuration
SAVE                            # Save database (RDB)
BGSAVE                          # Background save
"""

print(reference)


# STRING COMMANDS
SET key value [EX seconds]      # Set value with optional expiration
GET key                          # Get value
MSET key1 val1 key2 val2        # Set multiple key-value pairs
MGET key1 key2                   # Get multiple values
INCR key                         # Increment by 1
DECR key                         # Decrement by 1
APPEND key value                 # Append to string

# LIST COMMANDS
RPUSH key value                  # Push to right
LPUSH key value                  # Push to left
RPOP key                         # Pop from right
LPOP key                         # Pop from left
LLEN key                         # List length
LRANGE key start end             # Get range
LTRIM key start end              # Keep only range

# SET COMMANDS
SADD key member                  # Add member
REM key member                   # Remove member
SMEMBERS key                     # Get all members
SISMEMBER key member            # Check membership
SINTER key1 key2            

---
# Conclusion

You now have a comprehensive understanding of Redis from fundamentals to production-ready FastAPI integration!

### Next Steps:
1. **Practice** with the examples in this notebook
2. **Implement** caching in your applications
3. **Monitor** Redis performance in production
4. **Explore** advanced features like Lua scripting and clustering
5. **Deploy** with proper persistence and backup strategies

### Resources:
- Redis Official: https://redis.io
- Redis Python: https://github.com/redis/redis-py
- FastAPI: https://fastapi.tiangolo.com
- Redis Documentation: https://redis.io/docs